In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from tokenizer.bpe import ByteLevelBPETokenizer

In [3]:
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

tokenizer_paths = {
    "8K": ARTIFACT_DIR / "tokenizer_8192.json",
    "16K": ARTIFACT_DIR / "tokenizer_16384.json",
    "32K": ARTIFACT_DIR / "tokenizer_32768.json",
}

tokenizers = {}

for name, path in tokenizer_paths.items():
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.load(path)
    tokenizers[name] = tokenizer

print({name: len(tokenizer.vocab) for name, tokenizer in tokenizers.items()})

{'8K': 8192, '16K': 16384, '32K': 27638}


In [4]:
sample_text = """
Once upon a time, there was a little girl named Lily.
Lily had three small birds. One bird was red, one was blue,
and one was yellow. Lily loved her birds very much.
"""

In [5]:
for name, tokenizer in tokenizers.items():
    tokens = tokenizer.encode(sample_text)

    print(f"{name}:")
    print(f"  Tokens: {len(tokens)}")
    print(f"  IDs: {tokens[:30]}")
    print()

8K:
  Tokens: 44
  IDs: [10, 428, 435, 258, 396, 44, 399, 282, 258, 388, 475, 400, 402, 46, 10, 666, 362, 1476, 555, 1111, 46, 526, 416, 282, 671, 44, 725, 282, 939, 44]

16K:
  Tokens: 44
  IDs: [10, 428, 435, 258, 396, 44, 399, 282, 258, 388, 475, 400, 402, 46, 10, 666, 362, 1476, 555, 1111, 46, 526, 416, 282, 671, 44, 725, 282, 939, 44]

32K:
  Tokens: 44
  IDs: [10, 428, 435, 258, 396, 44, 399, 282, 258, 388, 475, 400, 402, 46, 10, 666, 362, 1476, 555, 1111, 46, 526, 416, 282, 671, 44, 725, 282, 939, 44]



In [6]:
results = {}

for name, tokenizer in tokenizers.items():
    tokens = tokenizer.encode(sample_text)

    results[name] = {
        "tokens": len(tokens),
        "characters": len(sample_text),
        "tokens_per_char": len(tokens) / len(sample_text),
        "chars_per_token": len(sample_text) / len(tokens),
    }

results

{'8K': {'tokens': 44,
  'characters': 167,
  'tokens_per_char': 0.2634730538922156,
  'chars_per_token': 3.7954545454545454},
 '16K': {'tokens': 44,
  'characters': 167,
  'tokens_per_char': 0.2634730538922156,
  'chars_per_token': 3.7954545454545454},
 '32K': {'tokens': 44,
  'characters': 167,
  'tokens_per_char': 0.2634730538922156,
  'chars_per_token': 3.7954545454545454}}

### Using general_text.txt for eval how different tokenizers are playing out

In [7]:
from pathlib import Path

EVAL_FILE = Path("../data/eval/general_text.txt")

eval_text = EVAL_FILE.read_text(encoding="utf-8")

print(f"Characters: {len(eval_text):,}")
print(eval_text[:300])

Characters: 2,950
Artificial intelligence systems are increasingly being used to analyze large collections of documents, summarize information, generate code, and assist with complex decision-making. A modern language model does not process raw characters directly. Instead, text is converted into a sequence of tokens


In [8]:
from tokenizer.bpe import ByteLevelBPETokenizer

ARTIFACT_DIR = Path("../artifacts")

tokenizer_paths = {
    "8K": ARTIFACT_DIR / "tokenizer_8192.json",
    "16K": ARTIFACT_DIR / "tokenizer_16384.json",
    "27.6K": ARTIFACT_DIR / "tokenizer_32768.json",
}

tokenizers = {}

for name, path in tokenizer_paths.items():
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.load(path)
    tokenizers[name] = tokenizer

print({
    name: len(tokenizer.vocab)
    for name, tokenizer in tokenizers.items()
})

{'8K': 8192, '16K': 16384, '27.6K': 27638}


In [9]:
results = {}

for name, tokenizer in tokenizers.items():
    tokens = tokenizer.encode(eval_text)

    results[name] = {
        "vocab_size": len(tokenizer.vocab),
        "characters": len(eval_text),
        "tokens": len(tokens),
        "tokens_per_char": len(tokens) / len(eval_text),
        "chars_per_token": len(eval_text) / len(tokens),
    }

results

{'8K': {'vocab_size': 8192,
  'characters': 2950,
  'tokens': 1078,
  'tokens_per_char': 0.36542372881355933,
  'chars_per_token': 2.7365491651205938},
 '16K': {'vocab_size': 16384,
  'characters': 2950,
  'tokens': 971,
  'tokens_per_char': 0.32915254237288133,
  'chars_per_token': 3.038105046343975},
 '27.6K': {'vocab_size': 27638,
  'characters': 2950,
  'tokens': 880,
  'tokens_per_char': 0.2983050847457627,
  'chars_per_token': 3.352272727272727}}

In [10]:
baseline_tokens = results["8K"]["tokens"]

for name, result in results.items():
    reduction = (
        1 - result["tokens"] / baseline_tokens
    ) * 100

    print(
        f"{name}: "
        f"{result['tokens']:,} tokens | "
        f"{reduction:.2f}% fewer tokens vs 8K"
    )

8K: 1,078 tokens | 0.00% fewer tokens vs 8K
16K: 971 tokens | 9.93% fewer tokens vs 8K
27.6K: 880 tokens | 18.37% fewer tokens vs 8K


In [11]:
sample = """
Artificial intelligence systems are increasingly being used
to analyze large collections of documents and generate code.
"""

for name, tokenizer in tokenizers.items():
    tokens = tokenizer.encode(sample)

    print(f"\n{name}")
    print("Token count:", len(tokens))
    print("Token IDs:", tokens)
    print("Decoded:", tokenizer.decode(tokens))

assert tokenizer.decode(tokens) == sample


8K
Token count: 42
Token IDs: [10, 65, 114, 116, 612, 461, 850, 3418, 1680, 259, 121, 349, 950, 115, 479, 4495, 3591, 460, 1009, 1134, 10, 2338, 418, 417, 121, 1763, 2579, 2910, 4067, 368, 373, 99, 490, 5522, 266, 2633, 840, 579, 100, 101, 46, 10]
Decoded: 
Artificial intelligence systems are increasingly being used
to analyze large collections of documents and generate code.


16K
Token count: 33
Token IDs: [10, 12038, 116, 612, 461, 850, 10079, 259, 121, 349, 11489, 479, 8467, 460, 1009, 1134, 10, 2338, 418, 417, 121, 1763, 2579, 10436, 368, 14633, 15343, 266, 2633, 840, 10389, 46, 10]
Decoded: 
Artificial intelligence systems are increasingly being used
to analyze large collections of documents and generate code.


27.6K
Token count: 27
Token IDs: [10, 20042, 19538, 850, 10079, 16840, 349, 11489, 479, 20804, 1009, 1134, 10, 2338, 418, 417, 121, 1763, 2579, 10436, 368, 20466, 266, 24631, 10389, 46, 10]
Decoded: 
Artificial intelligence systems are increasingly being used
to analyze 

In [12]:
for name, tokenizer in tokenizers.items():
    tokens = tokenizer.encode(sample)

    print(f"\n{name}")

    for token_id in tokens:
        token_bytes = tokenizer.vocab[token_id]
        print(
            token_id,
            repr(token_bytes.decode("utf-8", errors="replace"))
        )


8K
10 '\n'
65 'A'
114 'r'
116 't'
612 'if'
461 'ic'
850 'ial'
3418 ' intellig'
1680 'ence'
259 ' s'
121 'y'
349 'st'
950 'em'
115 's'
479 ' are'
4495 ' incre'
3591 'asing'
460 'ly'
1009 ' being'
1134 ' used'
10 '\n'
2338 'to'
418 ' an'
417 'al'
121 'y'
1763 'ze'
2579 ' large'
2910 ' collect'
4067 'ions'
368 ' of'
373 ' do'
99 'c'
490 'um'
5522 'ents'
266 ' and'
2633 ' gener'
840 'ate'
579 ' co'
100 'd'
101 'e'
46 '.'
10 '\n'

16K
10 '\n'
12038 'Ar'
116 't'
612 'if'
461 'ic'
850 'ial'
10079 ' intelligence'
259 ' s'
121 'y'
349 'st'
11489 'ems'
479 ' are'
8467 ' increasing'
460 'ly'
1009 ' being'
1134 ' used'
10 '\n'
2338 'to'
418 ' an'
417 'al'
121 'y'
1763 'ze'
2579 ' large'
10436 ' collections'
368 ' of'
14633 ' doc'
15343 'uments'
266 ' and'
2633 ' gener'
840 'ate'
10389 ' code'
46 '.'
10 '\n'

27.6K
10 '\n'
20042 'Art'
19538 'ific'
850 'ial'
10079 ' intelligence'
16840 ' sy'
349 'st'
11489 'ems'
479 ' are'
20804 ' increasingly'
1009 ' being'
1134 ' used'
10 '\n'
2338 'to'
418 ' an'

In [13]:
# Evaluate all tokenizer vocabulary sizes across the complete evaluation corpus

from pathlib import Path
import pandas as pd

EVAL_FILE = Path("../data/eval/multi_language_text.txt")
ARTIFACT_DIR = Path("../artifacts")

tokenizer_paths = {
    "8K": ARTIFACT_DIR / "tokenizer_8192.json",
    "16K": ARTIFACT_DIR / "tokenizer_16384.json",
    "27.6K": ARTIFACT_DIR / "tokenizer_32768.json",
}

# Load evaluation corpus
eval_text = EVAL_FILE.read_text(encoding="utf-8")

print(f"Evaluation corpus: {EVAL_FILE}")
print(f"Characters: {len(eval_text):,}")
print()

# Load tokenizers
from tokenizer.bpe import ByteLevelBPETokenizer

tokenizers = {}

for name, path in tokenizer_paths.items():
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.load(path)
    tokenizers[name] = tokenizer

# Evaluate each tokenizer
results = []

for name, tokenizer in tokenizers.items():
    tokens = tokenizer.encode(eval_text)

    results.append({
        "tokenizer": name,
        "vocab_size": len(tokenizer.vocab),
        "characters": len(eval_text),
        "tokens": len(tokens),
        "tokens_per_char": len(tokens) / len(eval_text),
        "chars_per_token": len(eval_text) / len(tokens),
    })

results_df = pd.DataFrame(results)

# Calculate token reduction relative to the 8K tokenizer
baseline_tokens = results_df.loc[
    results_df["tokenizer"] == "8K",
    "tokens"
].iloc[0]

results_df["token_reduction_vs_8K_%"] = (
    1 - results_df["tokens"] / baseline_tokens
) * 100

results_df

Evaluation corpus: ../data/eval/multi_language_text.txt
Characters: 11,684



,tokenizer,vocab_size,characters,tokens,tokens_per_char,chars_per_token,token_reduction_vs_8K_%
0,8K,8192,11684,6821,0.583790,1.712945,0.000000
1,16K,16384,11684,6356,0.543992,1.838263,6.817182
2,27.6K,27638,11684,5878,0.503081,1.987751,13.824952


In [16]:
# Evaluate tokenizer efficiency by category

from pathlib import Path
import re
import pandas as pd

from tokenizer.bpe import ByteLevelBPETokenizer


# Paths
EVAL_FILE = Path("../data/eval/multi_language_text.txt")
ARTIFACT_DIR = Path("../artifacts")

tokenizer_paths = {
    "8K": ARTIFACT_DIR / "tokenizer_8192.json",
    "16K": ARTIFACT_DIR / "tokenizer_16384.json",
    "27.6K": ARTIFACT_DIR / "tokenizer_32768.json",
}


# Load evaluation corpus
eval_text = EVAL_FILE.read_text(encoding="utf-8")


# Split corpus using "# Section Name" headings
sections = re.split(r"^# (.+)$", eval_text, flags=re.MULTILINE)

categories = {}

for i in range(1, len(sections), 2):
    category = sections[i].strip()
    text = sections[i + 1].strip()

    if text:
        categories[category] = text


print(f"Categories found: {len(categories)}")
print(list(categories.keys()))


# Load tokenizers
tokenizers = {}

for name, path in tokenizer_paths.items():
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.load(path)
    tokenizers[name] = tokenizer


# Evaluate each category with each tokenizer
results = []

for category, text in categories.items():

    for tokenizer_name, tokenizer in tokenizers.items():

        tokens = tokenizer.encode(text)

        results.append({
            "category": category,
            "tokenizer": tokenizer_name,
            "characters": len(text),
            "tokens": len(tokens),
            "tokens_per_char": len(tokens) / len(text),
            "chars_per_token": len(text) / len(tokens),
        })


results_df = pd.DataFrame(results)


# Display complete results
results_df.head(100)

Categories found: 23
['English Prose', 'Technical Text', 'Python', 'File path used by the training pipeline:', 'A simple list comprehension can transform records:', 'Unicode strings can contain characters from many writing systems:', 'Long identifiers are common in real software:', 'URLs and API endpoints are also common model inputs:', 'SQL', 'URLs and Paths', 'Numbers, Dates and Measurements', 'Rare and Long Words', 'Hindi', 'Japanese', 'Chinese', 'Emoji and Symbols', 'Mixed Hindi-English', 'Code and Identifiers', 'Long identifiers and variable names', 'Regular expressions', 'JSON', 'Shell commands', 'Mixed Technical Content']


,category,tokenizer,characters,tokens,tokens_per_char,chars_per_token
0,English Prose,8K,1108,320,0.288809,3.462500
1,English Prose,16K,1108,269,0.242780,4.118959
2,English Prose,27.6K,1108,234,0.211191,4.735043
3,Technical Text,8K,1178,369,0.313243,3.192412
4,Technical Text,16K,1178,328,0.278438,3.591463
...,...,...,...,...,...,...
64,Shell commands,16K,230,122,0.530435,1.885246
65,Shell commands,27.6K,230,112,0.486957,2.053571
66,Mixed Technical Content,8K,1366,1070,0.783309,1.276636
67,Mixed Technical Content,16K,1366,979,0.716691,1.395301


In [17]:
# Compare tokens per character across tokenizer sizes

comparison = results_df.pivot(
    index="category",
    columns="tokenizer",
    values="tokens_per_char"
)

comparison

tokenizer,16K,27.6K,8K
category,,,
A simple list comprehension can transform records:,0.462500,0.450000,0.487500
Chinese,0.182927,0.182927,0.201220
Code and Identifiers,0.609160,0.522137,0.622901
Emoji and Symbols,1.231343,1.227612,1.272388
English Prose,0.242780,0.211191,0.288809
File path used by the training pipeline:,0.567568,0.567568,0.648649
Hindi,1.089136,1.089136,1.101671
JSON,0.639785,0.569892,0.666667
Japanese,0.208333,0.208333,0.222222


In [18]:
# Calculate token reduction relative to the 8K tokenizer for each category

comparison_reduction = comparison.copy()

for category in comparison.index:
    baseline = comparison.loc[category, "8K"]

    for tokenizer_name in ["16K", "27.6K"]:
        comparison_reduction.loc[category, tokenizer_name] = (
            1
            - comparison.loc[category, tokenizer_name] / baseline
        ) * 100

comparison_reduction = comparison_reduction.drop(columns="8K")

comparison_reduction

tokenizer,16K,27.6K
category,,
A simple list comprehension can transform records:,5.128205,7.692308
Chinese,9.090909,9.090909
Code and Identifiers,2.205882,16.176471
Emoji and Symbols,3.225806,3.519062
English Prose,15.937500,26.875000
File path used by the training pipeline:,12.500000,12.500000
Hindi,1.137800,1.137800
JSON,4.032258,14.516129
Japanese,6.250000,6.250000
